## Imports

In [50]:
import os
import uuid
import asyncio
import dotenv
import openai
import pandas as pd
from datetime import datetime, timezone
from qdrant_client import QdrantClient
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langsmith import Client
from langsmith.schemas import RunTypeEnum
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall
from ragas.metrics.collections import Faithfulness, AnswerRelevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.dataset_schema import SingleTurnSample

dotenv.load_dotenv()

/var/folders/qr/_t3nsks152d7njyq9d_kcqx40000gn/T/ipykernel_38964/3038694843.py:12: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall
/var/folders/qr/_t3nsks152d7njyq9d_kcqx40000gn/T/ipykernel_38964/3038694843.py:12: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall


True

## Load Eval Dataset from LangSmith

In [ ]:
DATASET_NAME = "hephaestus-rag-eval"

ls_client = Client()
examples = list(ls_client.list_examples(dataset_name=DATASET_NAME))

sample_types = [e.metadata.get('sample_type') for e in examples]
print(f"Loaded {len(examples)} examples — single: {sample_types.count('single')}  multi: {sample_types.count('multi')}")

## RAG Pipeline

In [51]:
qdrant_client = QdrantClient(url="http://localhost:6333")
COLLECTION_NAME = "cm_interventions"


def embed_text(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(input=text, model=model)
    return response.data[0].embedding


def retrieve(query, top_k=5):
    vector = embed_text(query)
    hits = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=vector,
        limit=top_k,
    ).points
    return hits


def format_context(hits):
    parts = []
    for h in hits:
        p = h.payload
        parts.append(
            f"ID: {p.get('id')}\nMachine: {p.get('machine')}\n"
            f"Date: {p.get('date_start')}\nSummary: {p.get('summary')}"
        )
    return "\n\n---\n\n".join(parts)


RAG_PROMPT = """\
You are a maintenance assistant. Use the intervention records below to answer the question.
If the records do not contain relevant information, say \"I don't know\".
Be concise. Do not use markdown formatting.

Question: {question}

Records:
{context}
"""


def generate_answer(question, context):
    prompt = RAG_PROMPT.format(question=question, context=context)
    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content.strip()


def rag_pipeline(question, top_k=5):
    hits = retrieve(question, top_k)
    context = format_context(hits)
    answer = generate_answer(question, context)
    retrieved_ids = [h.payload.get('id') for h in hits]
    return answer, context, retrieved_ids

/Users/jooaobrum/Library/CloudStorage/GoogleDrive-joao.paulo.brum14@gmail.com/My Drive/Projetos Pessoais/Projetos de Estudo/end2end-ai-engineering-bootcamp/hephaestus-agentic-maintenance/.venv/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:280: UserWarning: Qdrant client version 1.17.1 is incompatible with server version 1.12.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


## Run Pipeline on Eval Set

In [ ]:
results = []

for ex in examples:
    question     = ex.inputs["question"]
    ground_truth = ex.outputs["ground_truth"]
    expected_ids = ex.outputs["chunks_id"]
    sample_type  = ex.metadata.get("sample_type")

    answer, context, retrieved_ids = rag_pipeline(question)

    results.append({
        "example_id":    str(ex.id),
        "sample_type":   sample_type,
        "question":      question,
        "answer":        answer,
        "ground_truth":  ground_truth,
        "context":       context,
        "retrieved_ids": retrieved_ids,
        "expected_ids":  expected_ids,
    })
    print(f"✓ [{sample_type}] {question[:70]}")

df_results = pd.DataFrame(results)
print(f"\nDone. {len(df_results)} rows.")

In [71]:
df_results = pd.read_csv("results.csv", sep = ';')

In [72]:
df_results = df_results.drop("Unnamed: 0", axis = 1)

## RAGAS Evaluation

In [73]:
import json as _json
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall
from ragas.metrics.collections import Faithfulness, AnswerRelevancy

# Must use AsyncOpenAI — sync client breaks async scoring
openai_async_client = AsyncOpenAI()
ragas_llm = llm_factory("gpt-4o-mini", client=openai_async_client)
ragas_emb = embedding_factory("openai", model="text-embedding-3-small", client=openai_async_client)


def _to_list(val):
    """DataFrame stores lists as strings after CSV round-trip."""
    return _json.loads(val.replace("'", '"')) if isinstance(val, str) else val


async def ragas_context_precision_id(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=_to_list(run["retrieved_ids"]),
        reference_context_ids=_to_list(example["chunks_id"]),
    )
    return await IDBasedContextPrecision().single_turn_ascore(sample)


async def ragas_context_recall_id(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=_to_list(run["retrieved_ids"]),
        reference_context_ids=_to_list(example["chunks_id"]),
    )
    return await IDBasedContextRecall().single_turn_ascore(sample)


async def ragas_faithfulness(run, example):
    result = await Faithfulness(llm=ragas_llm).ascore(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=[run["context"]],
    )
    return result.value


async def ragas_answer_relevancy(run, example):
    result = await AnswerRelevancy(llm=ragas_llm, embeddings=ragas_emb).ascore(
        user_input=run["question"],
        response=run["answer"],
    )
    return result.value


METRIC_FNS = {
    "context_precision_id": ragas_context_precision_id,
    "context_recall_id":    ragas_context_recall_id,
    "faithfulness":         ragas_faithfulness,
    "answer_relevancy":     ragas_answer_relevancy,
}


async def score_row(i, n, run, example):
    scores = {}
    for name, fn in METRIC_FNS.items():
        try:
            scores[name] = await fn(run, example)
        except Exception as e:
            print(f"  ⚠ {name}: {e}")
            scores[name] = None
    return scores


async def run_eval(df, batch_size=20):
    n = len(df)
    rows = [(i, row.to_dict(), {"chunks_id": row["expected_ids"]}) for i, (_, row) in enumerate(df.iterrows())]
    all_scores = []

    for start in range(0, n, batch_size):
        batch = rows[start : start + batch_size]
        print(f"\n--- Batch {start//batch_size + 1} ({start+1}–{min(start+batch_size, n)} of {n}) ---")
        batch_scores = await asyncio.gather(*[
            score_row(i, n, run, example) for i, run, example in batch
        ])
        all_scores.extend(batch_scores)

    return pd.DataFrame(all_scores)


df_scores = await run_eval(df_results, batch_size=20)
print(df_scores)

/var/folders/qr/_t3nsks152d7njyq9d_kcqx40000gn/T/ipykernel_38964/1017940128.py:5: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall
/var/folders/qr/_t3nsks152d7njyq9d_kcqx40000gn/T/ipykernel_38964/1017940128.py:5: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall
/var/folders/qr/_t3nsks152d7njyq9d_kcqx40000gn/T/ipykernel_38964/1017940128.py:11: DeprecationWarning: Importing embedding_factory from ragas.embeddings is deprecated. Import directly from ragas.embeddings.base or use mo


--- Batch 1 (1–20 of 116) ---
  ⚠ faithfulness: <failed_attempts>

<generation number="1">
<exception>
    The output is incomplete due to a max_tokens length limit.
</exception>
<completion>
    ChatCompletion(id='chatcmpl-DOjDz4NqC9JP4Ej1u6AIWkFStVMxI', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n    "statements": [\n        {\n            "statement": "The confirmed causes and repair actions for machine HX-350 E-001 (PS-101 low pressure) are detailed.",\n            "reason": "The context provides detailed information about the causes and repair actions for machine HX-350 E-001.",\n            "verdict": 1\n        },\n        {\n            "statement": "On 2022-11-01, the cause of the low pressure was coupling misalignment after recent maintenance, and the coupling was worn beyond limits.",\n            "reason": "The context explicitly states that on 2022-11-01, the cause was coupling misalignment and confirms the cou

## Results Breakdown

In [74]:
METRIC_COLS = list(METRIC_FNS.keys())

df_scores["sample_type"] = df_results["sample_type"].values

print("=== Overall ===")
print(df_scores[METRIC_COLS].mean().round(3).to_string())

print("\n=== By sample_type ===")
print(df_scores.groupby("sample_type")[METRIC_COLS].mean().round(3).to_string())

=== Overall ===
context_precision_id    0.290
context_recall_id       0.991
faithfulness            0.900
answer_relevancy        0.627

=== By sample_type ===
             context_precision_id  context_recall_id  faithfulness  answer_relevancy
sample_type                                                                         
multi                       0.417              0.979         0.890             0.618
single                      0.200              1.000         0.904             0.634


## Upload Results to LangSmith

In [75]:
project_name = os.getenv("LANGSMITH_PROJECT")

for i, row in df_results.iterrows():
    scores = df_scores.iloc[i]
    run_id = uuid.uuid4()
    now    = datetime.now(timezone.utc)

    ls_client.create_run(
        id=run_id,
        name="rag-eval",
        run_type=RunTypeEnum.chain,
        project_name=project_name,
        inputs={"question": row["question"], "context": row["context"]},
        outputs={"answer": row["answer"]},
        reference_example_id=row["example_id"],
        start_time=now,
        end_time=now,
    )

    for metric in METRIC_COLS:
        score = scores[metric]
        if pd.notna(score):
            ls_client.create_feedback(
                run_id=run_id,
                key=metric,
                score=float(score),
            )

print(f"Uploaded {len(df_results)} runs with RAGAS scores to '{project_name}'")

Uploaded 116 runs with RAGAS scores to 'hephaestus-agentic-maintenance'


In [76]:
from concurrent.futures import ThreadPoolExecutor

project_name = os.getenv("LANGSMITH_PROJECT")
METRIC_COLS = ["context_precision_id", "context_recall_id", "faithfulness", "answer_relevancy"]

# Create an experiment project linked to the dataset
dataset = ls_client.read_dataset(dataset_name=DATASET_NAME)
experiment_name = f"rag-eval-baseline"

experiment = ls_client.create_project(
    experiment_name,
    description="Baseline RAG eval — RAGAS metrics (IDBasedPrecision/Recall, Faithfulness, AnswerRelevancy)",
    reference_dataset_id=dataset.id,
    upsert=True,
)
print(f"Experiment: {experiment_name} (linked to dataset {DATASET_NAME})")


def upload_row(args):
    i, row = args
    scores = df_scores.iloc[i]
    run_id = uuid.uuid4()
    now    = datetime.now(timezone.utc)

    ls_client.create_run(
        id=run_id,
        name="rag-eval",
        run_type=RunTypeEnum.chain,
        project_name=experiment_name,
        inputs={"question": row["question"], "context": row["context"]},
        outputs={"answer": row["answer"]},
        reference_example_id=row["example_id"],
        start_time=now,
        end_time=now,
    )

    for metric in METRIC_COLS:
        score = scores[metric]
        if pd.notna(score):
            ls_client.create_feedback(run_id=run_id, key=metric, score=float(score))

    return i


with ThreadPoolExecutor(max_workers=10) as executor:
    for i in executor.map(upload_row, df_results.iterrows()):
        print(f"✓ uploaded row {i+1}/{len(df_results)}")

print(f"Done. Uploaded {len(df_results)} runs to experiment '{experiment_name}'")

Experiment: rag-eval-baseline (linked to dataset rag-evaluation-dataset)
✓ uploaded row 1/116
✓ uploaded row 2/116
✓ uploaded row 3/116
✓ uploaded row 4/116
✓ uploaded row 5/116
✓ uploaded row 6/116
✓ uploaded row 7/116
✓ uploaded row 8/116
✓ uploaded row 9/116
✓ uploaded row 10/116
✓ uploaded row 11/116
✓ uploaded row 12/116
✓ uploaded row 13/116
✓ uploaded row 14/116
✓ uploaded row 15/116
✓ uploaded row 16/116
✓ uploaded row 17/116
✓ uploaded row 18/116
✓ uploaded row 19/116
✓ uploaded row 20/116
✓ uploaded row 21/116
✓ uploaded row 22/116
✓ uploaded row 23/116
✓ uploaded row 24/116
✓ uploaded row 25/116
✓ uploaded row 26/116
✓ uploaded row 27/116
✓ uploaded row 28/116
✓ uploaded row 29/116
✓ uploaded row 30/116
✓ uploaded row 31/116
✓ uploaded row 32/116
✓ uploaded row 33/116
✓ uploaded row 34/116
✓ uploaded row 35/116
✓ uploaded row 36/116
✓ uploaded row 37/116
✓ uploaded row 38/116
✓ uploaded row 39/116
✓ uploaded row 40/116
✓ uploaded row 41/116
✓ uploaded row 42/116
✓ uploaded r